# UAS Risk Analysis Visualization

## Setup

In [1]:
!python -m pip install -q geopandas seaborn s3fs pyarrow fiona

### Imports

In [15]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import zipfile
from glob import glob
import subprocess

### Configurations

In [3]:
sns.set_style('darkgrid')

## Load UAS Risk Analysis Results

In [4]:
risk_scores_df = pd.read_csv('s3://endurasoft-dev-risk-framework/analysis/uas_risk_scores/consolidated/uas_risk_scores_csv/uas_risk_scores.csv', low_memory=False)
risk_scores_df.head()

,objectid,ceiling,unit,map_eff,last_edit,latitude,longitude,globalid,arpt_count,apt1_faaid,...,apt4_enabled,apt5_enabled,shape__length,shape__area,shape__area_2,shape__length_2,wkt,flight_intersections,uas_sightings_intersections,risk_score
0,7,400,Feet,11/28/2024,10/11/2017,28.704170,-81.204180,bf777832-0a72-42eb-b6c4-57e1c864df1d,1,SFB,...,NaN,NaN,NaN,NaN,0.000069,0.033333,POLYGON Z ((-81.2083407217845 28.7000043885012...,0,0,0.0
1,24,200,Feet,11/28/2024,4/27/2017,28.712505,-96.195850,c26447b0-6cb4-47b3-bcb1-47b78ace8851,1,PSX,...,NaN,NaN,NaN,NaN,0.000069,0.033333,POLYGON Z ((-96.2000103871126 28.7083377237965...,0,0,0.0
2,63,400,Feet,11/28/2024,10/11/2017,28.720839,-81.279175,cb05ee17-6048-4f0c-92d1-610fe7fb63cc,1,SFB,...,NaN,NaN,NaN,NaN,0.000069,0.033333,POLYGON Z ((-81.2833407374426 28.7166710590919...,0,0,0.0
3,67,300,Feet,11/28/2024,10/11/2017,28.720839,-81.237510,0e7fad4e-9cda-4364-9cbe-0b34516967be,1,SFB,...,NaN,NaN,NaN,NaN,0.000069,0.033333,POLYGON Z ((-81.2416740619659 28.7166710590919...,0,0,0.0
4,69,300,Feet,11/28/2024,10/11/2017,28.720839,-81.220840,5a5f7a74-07cc-49cb-86b9-50c323b5609f,1,SFB,...,NaN,NaN,NaN,NaN,0.000069,0.033333,POLYGON Z ((-81.2250073923752 28.7166710590919...,0,0,0.0


In [5]:
risk_scores_df.shape

(376569, 49)

In [6]:
# Rename long target columns (columns with more than 10 characters are truncated)
risk_scores_df.rename({'flight_intersections': 'flight_int', 'uas_sightings_intersections': 'uas_int'}, axis=1, inplace=True)

## Convert to GeoPandas and Save Shapefile

In [7]:
# Convert risk scores dataframe to geopandas
risk_scores_gdf = gpd.GeoDataFrame(
    risk_scores_df,
    geometry=gpd.GeoSeries.from_wkt(risk_scores_df['wkt'])
)

In [9]:
shapefile_dir = 'uas_risk_scores_shapefile'
os.makedirs(shapefile_dir, exist_ok=True)
risk_scores_gdf.to_file(os.path.join(shapefile_dir, 'uas_risk_scores.shp'), driver='ESRI Shapefile', engine='fiona', crs='EPSG:4326')

/tmp/ipykernel_13144/343700671.py:3: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  risk_scores_gdf.to_file(os.path.join(shapefile_dir, 'uas_risk_scores.shp'), driver='ESRI Shapefile', engine='fiona', crs='EPSG:4326')


In [13]:
glob(os.path.join(shapefile_dir, '*.*'))

['uas_risk_scores_shapefile/uas_risk_scores.shx',
 'uas_risk_scores_shapefile/uas_risk_scores.shp',
 'uas_risk_scores_shapefile/uas_risk_scores.prj',
 'uas_risk_scores_shapefile/uas_risk_scores.dbf',
 'uas_risk_scores_shapefile/uas_risk_scores.cpg']

In [14]:
# Create a zip file
with zipfile.ZipFile('uas_risk_scores_shapefile.zip', 'w', zipfile.ZIP_DEFLATED) as zipf:
    # Add all shapefile components to zip
    for path in glob(os.path.join(shapefile_dir, '*.*')):
        zipf.write(path)

In [16]:
# Upload to s3
subprocess.check_call('aws s3 cp uas_risk_scores_shapefile.zip s3://endurasoft-dev-risk-framework/analysis/uas_risk_scores/viz/uas_risk_scores_shapefile.zip'.split())

upload: ./uas_risk_scores_shapefile.zip to s3://endurasoft-dev-risk-framework/analysis/uas_risk_scores/viz/uas_risk_scores_shapefile.zip


0